# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:** 32 × 32 = **1024 valores**. Al pasar a escala de grises hay un solo canal, así que la imagen 32×32 se aplana en un vector de 1024 componentes.

## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización  --> RESUELTO
# Los pixeles estan en [0, 255]; dividimos entre 255 para llevarlos a [0, 1].
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())

In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura  --> RESUELTO
model = keras.Sequential([
    keras.Input(shape=(32, 32)),      # imagen grayscale 32x32
    layers.Flatten(),                 # -> 1024 valores
    layers.Dense(128, activation="relu"),
    layers.Dense(10)                  # 10 clases (logits, sin activacion)
])

model.summary()

In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** una capa `Dense` espera un vector 1D por ejemplo, pero la imagen entra como una matriz 2D (32×32). `Flatten()` convierte esa matriz en un vector de 1024 valores para poder conectarlo a la capa densa.

**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:** porque CIFAR-10 tiene **10 clases**. Cada neurona produce el *logit* (puntaje) de una clase; luego Softmax los convierte en probabilidades.

**4.** ¿Qué función cumple `ReLU`?

**Respuesta:** introduce **no linealidad** ( `ReLU(x)=max(0,x)` ). Sin una activación no lineal, apilar capas densas equivaldría a una sola transformación lineal; ReLU permite que la red aprenda relaciones no lineales entre los píxeles y las clases.

---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento  --> RESUELTO
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:** porque la última capa `Dense(10)` **no** tiene Softmax, así que entrega *logits* (valores sin normalizar). `from_logits=True` le indica a la pérdida que aplique internamente el log-softmax de forma numéricamente estable, en lugar de asumir que recibe probabilidades ya normalizadas.

---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento  --> RESUELTO
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)

### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:** una pasada completa por **todo** el conjunto de entrenamiento. En cada época la red ve todos los ejemplos una vez (en mini-batches) y actualiza sus pesos.

**7.** ¿Qué representa `batch_size=64`?

**Respuesta:** el número de ejemplos que se procesan juntos antes de cada actualización de pesos. Con 64, los gradientes se promedian sobre 64 imágenes por paso; es un equilibrio entre la estabilidad del gradiente y el uso de memoria/velocidad.

## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:** sí, es esperable algo de overfitting en esta red *fully-connected* sobre CIFAR-10. Suele verse que la **accuracy de entrenamiento sigue subiendo** mientras la **de validación se estanca** (alrededor de 0.40–0.50) e incluso la `val_loss` empieza a subir en las últimas épocas. Esa brecha creciente entre train y validation es la señal típica de sobreajuste.

---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación  --> RESUELTO
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")

### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:** **más difícil**. En Fashion-MNIST esta misma red fully-connected llega a ~87–89 %, mientras que en CIFAR-10 en escala de grises ronda apenas **~40–50 %**. CIFAR-10 contiene objetos naturales con fondos variados, poses y colores (que además eliminamos al pasar a gris), lo que hace mucho más difícil separar las clases con una red que no aprovecha la estructura espacial.

---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades  --> RESUELTO
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad  --> RESUELTO
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])

In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:** Softmax normaliza los logits dividiendo cada exponencial entre la suma de todas las exponenciales, de modo que el resultado es una **distribución de probabilidad** sobre las 10 clases: todos los valores quedan en [0,1] y su suma es exactamente 1 (pequeñas diferencias se deben solo a redondeo de punto flotante).

---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta:** alrededor de **40–50 %** en test (el valor exacto aparece al ejecutar la celda de evaluación). Es muy superior al azar (10 %), pero modesto por tratarse de una red sin convoluciones sobre imágenes naturales en gris.

### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:** las clases de **animales con silueta parecida** (`cat`, `dog`, `deer`, `bird`) y los **vehículos entre sí** (`automobile`, `truck`). Al quitar el color y aplanar la imagen, estas categorías comparten patrones de intensidad muy similares.

### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante podría perderse?

**Respuesta:** se pierde la **estructura espacial**: la relación de vecindad entre píxeles (qué píxel está al lado de cuál), los bordes, las formas y los patrones locales. Para la red densa el píxel (0,0) y el (31,31) son dos entradas independientes sin noción de "cercanía".

### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial?

**Respuesta:** una **Red Neuronal Convolucional (CNN)**, que usa filtros locales que recorren la imagen y detectan bordes/formas respetando la vecindad de los píxeles.

---

## Para pensar

Nuestro modelo hace:
```text
32×32 → Flatten → 1024 valores → Dense → ReLU → Dense(10)
```
Una imagen tiene **estructura espacial**. La respuesta a cómo aprovecharla es: **Convolutional Neural Networks (CNNs)**.